In [1]:
import sys
from pathlib import Path
import torch

script_dir = Path.cwd()
sys.path.insert(0, str(script_dir.parent.parent))

from config import PROJECT_ROOT
from model import SSLModel
from data_split import create_train_val_split
from extract_XLSR_feature import extract_features_from_csv
from visualization import plot_training_curves, plot_dataset_comparison
from dataset import create_dataloaders
from train import train
import numpy as np

/opt/anaconda3/envs/madress/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
DATASET_NAME = "Pitt-Denoiser"
data_version = "denoised"

In [ ]:
# RAW_AUDIO_DIR = PROJECT_ROOT / f"data/denoised/{DATASET_NAME}"
RAW_AUDIO_DIR = PROJECT_ROOT / f"data/{data_version}/{DATASET_NAME}"

XLSR_FEATURES_DIR = PROJECT_ROOT / f"data/processed/{DATASET_NAME}_xlsr_features"
FEATURE_DIR_NAME = f"{DATASET_NAME}_xlsr_features"
MODEL_OUTPUT_DIR = PROJECT_ROOT / f"models/{DATASET_NAME}_xlsr_multi_seed"

In [ ]:
if torch.cuda.is_available():
      device = torch.device('cuda')
      accelerator = 'gpu'
elif torch.backends.mps.is_available():
      device = torch.device('mps')
      accelerator = 'mps'
else:
      device = torch.device('cpu')
      accelerator = 'cpu'
print(f"Using {device}")

## Step 1: Train/Validation Set Split

In [ ]:
# Define CSV paths
TRAIN_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-xlsr-train.csv"
VAL_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-xlsr-val.csv"

# Create CSV files if they don't exist
if not TRAIN_CSV.exists() or not VAL_CSV.exists():
    TRAIN_CSV, VAL_CSV = create_train_val_split(
        raw_audio_dir=RAW_AUDIO_DIR,
        dataset_name=DATASET_NAME,
        feature_dir_name=FEATURE_DIR_NAME,
        xlsr=True
    )


## Step 2: Extract XLSR Features

In [ ]:
ssl_model = SSLModel(device, freeze_xlsr=True)

extract_features_from_csv(
    csv_path=TRAIN_CSV,
    split_name="Train Set",
    raw_audio_dir=RAW_AUDIO_DIR,
    xlsr_features_dir=XLSR_FEATURES_DIR,
    device=device,
    ssl_model=ssl_model,
)

extract_features_from_csv(
    csv_path=VAL_CSV,
    split_name="Val Set",
    raw_audio_dir=RAW_AUDIO_DIR,
    xlsr_features_dir=XLSR_FEATURES_DIR,
    device=device,
    ssl_model=ssl_model,
)


## Step 4: Create Data Loaders


In [ ]:
train_loader = create_dataloaders(data_csv=TRAIN_CSV, xlsr=True)

val_loader = create_dataloaders(data_csv=VAL_CSV, xlsr=True)

## Step 5: Define Training Function and Model


In [ ]:
all_results = {
    'seeds': [], 
    'val_accs': [], 
    'val_losses': [], 
    'control_accs': [], 
    'dementia_accs': [], 
    'f1_scores': []
}

### 1st Random Seed = 21


In [ ]:
seed, metrics, history = train(seed=21, 
                               train_loader=train_loader, 
                               val_loader=val_loader, 
                               output_dir=MODEL_OUTPUT_DIR, 
                               device=device, 
                               xlsr=True)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

# Plot training curves
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed}'
)

### 2nd Random Seed = 42

In [ ]:
seed, metrics, history = train(seed=42, 
                               train_loader=train_loader, 
                               val_loader=val_loader, 
                               output_dir=MODEL_OUTPUT_DIR, 
                               device=device, 
                               xlsr=True)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

# Plot training curves
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed}'
)

### 3rd Random Seed = 84

In [ ]:
seed, metrics, history = train(seed=84, 
                               train_loader=train_loader, 
                               val_loader=val_loader, 
                               output_dir=MODEL_OUTPUT_DIR, 
                               device=device, 
                               xlsr=True)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

# Plot training curves
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed}'
)

### 4th Random Seed = 168

In [ ]:
seed, metrics, history = train(seed=168, 
                               train_loader=train_loader, 
                               val_loader=val_loader, 
                               output_dir=MODEL_OUTPUT_DIR, 
                               device=device, 
                               xlsr=True)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

# Plot training curves
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed}'
)

### 5th Random Seed = 336

In [ ]:
seed, metrics, history = train(seed=336, 
                               train_loader=train_loader, 
                               val_loader=val_loader, 
                               output_dir=MODEL_OUTPUT_DIR, 
                               device=device, 
                               xlsr=True)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

# Plot training curves
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed}'
)

## Step 6: Summary of Results

In [ ]:
seeds = all_results['seeds']
val_accs = [acc*100 for acc in all_results['val_accs']]
val_losses = all_results['val_losses']
control_accs = [acc*100 for acc in all_results['control_accs']]
dementia_accs = [acc*100 for acc in all_results['dementia_accs']]
f1_scores = all_results['f1_scores']

# Calculate statistics (mean and std)
mean_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

mean_loss = np.mean(val_losses)
std_loss = np.std(val_losses, ddof=1)

mean_control_acc = np.mean(control_accs)
std_control_acc = np.std(control_accs, ddof=1)

mean_dementia_acc = np.mean(dementia_accs)
std_dementia_acc = np.std(dementia_accs, ddof=1)

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores, ddof=1)

print(f"\nMean Validation Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"Mean Validation Loss: {mean_loss:.4f} ± {std_loss:.4f}")
print(f"Mean Control Accuracy: {mean_control_acc:.2f}% ± {std_control_acc:.2f}%")
print(f"Mean Dementia Accuracy: {mean_dementia_acc:.2f}% ± {std_dementia_acc:.2f}%")
print(f"Mean F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")

print(f"\nDetailed Results:")
for i, seed in enumerate(seeds):
    print(f"Seed {seed}: Acc={val_accs[i]:.2f}%, Loss={val_losses[i]:.4f}, "
          f"Control Acc={control_accs[i]:.2f}%, Dementia Acc={dementia_accs[i]:.2f}%, F1={f1_scores[i]:.4f}")

## Step 7: Test Best Model on Multiple Datasets


In [ ]:
# Find the best model: highest accuracy, then lowest loss if tied
max_acc = max(all_results['val_accs'])
max_acc_indices = [i for i, acc in enumerate(all_results['val_accs']) if acc == max_acc]

# If multiple models have the same max accuracy, choose the one with lowest loss
if len(max_acc_indices) > 1:
    print(f"Multiple models with accuracy {max_acc*100:.2f}%, selecting one with lowest loss")
    best_idx = min(max_acc_indices, key=lambda i: all_results['val_losses'][i])
else:
    best_idx = max_acc_indices[0]

# Get best model info
BEST_SEED = all_results['seeds'][best_idx]
BEST_VAL_ACC = all_results['val_accs'][best_idx] * 100
BEST_VAL_LOSS = all_results['val_losses'][best_idx]
BEST_F1 = all_results['f1_scores'][best_idx]
BEST_MODEL_PATH = MODEL_OUTPUT_DIR / f"seed_{BEST_SEED}" / "best.pth"

In [ ]:
# Load the best model
from model import AD_XLSR_Model
from test import test_on_dataset, test_on_dataset_with_val_csv

# Create model with XLSR configuration
test_model = AD_XLSR_Model()

# Load checkpoint (state_dict was saved directly)
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
test_model.load_state_dict(checkpoint)
test_model = test_model.to(device)
test_model.eval()

# Define reference validation CSV for consistent testing across datasets
# Use Pitt-Best validation set as the reference for all three datasets
REFERENCE_VAL_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-xlsr-val.csv"
print(f"Reference VAL_CSV: {REFERENCE_VAL_CSV}")

In [ ]:
# Test on Pitt dataset (using same validation set as Pitt-Best)
pitt_results = test_on_dataset_with_val_csv(
    reference_val_csv=REFERENCE_VAL_CSV,
    dataset_name="Pitt",
    model=test_model,
    device=device,
    ssl_model=ssl_model,
    raw_audio_dir="data/raw/Pitt"
)

In [ ]:
# Test on Pitt-Best dataset (using validation set)
pitt_best_results = test_on_dataset_with_val_csv(
    reference_val_csv=REFERENCE_VAL_CSV,
    dataset_name="Pitt-Best",
    model=test_model,
    device=device,
    ssl_model=ssl_model,
    raw_audio_dir="data/denoised/Pitt-Best"
)

In [ ]:
# Test on Pitt-residual-Best dataset (using same validation set as Pitt-Best)
pitt_residual_results = test_on_dataset_with_val_csv(
    reference_val_csv=REFERENCE_VAL_CSV,
    dataset_name="Pitt-residual-Best",
    model=test_model,
    device=device,
    ssl_model=ssl_model,
    raw_audio_dir="data/residual/Pitt-residual-Best"
)

In [ ]:
# Validate that all three datasets use the same session_id
import pandas as pd

pitt_val_df = pd.read_csv(PROJECT_ROOT / "data/processed/Pitt-xlsr-val.csv")
pitt_best_val_df = pd.read_csv(PROJECT_ROOT / "data/processed/Pitt-Best-xlsr-val.csv")
pitt_residual_val_df = pd.read_csv(PROJECT_ROOT / "data/processed/Pitt-residual-Best-xlsr-val.csv")

# Get session_id sets
pitt_sessions = set(pitt_val_df['session_id'])
pitt_best_sessions = set(pitt_best_val_df['session_id'])
pitt_residual_sessions = set(pitt_residual_val_df['session_id'])

# Check consistency
print("=== Validation Set Consistency Check ===")
print(f"Pitt sessions: {len(pitt_sessions)}")
print(f"Pitt-Best sessions: {len(pitt_best_sessions)}")
print(f"Pitt-residual-Best sessions: {len(pitt_residual_sessions)}")
print()

if pitt_sessions == pitt_best_sessions == pitt_residual_sessions:
    print("✓ All three datasets use the SAME validation set!")
    print(f"  Total validation samples: {len(pitt_sessions)}")
    
    # Show sample session_ids
    sample_ids = sorted(list(pitt_sessions))[:5]
    print(f"  Sample session_ids: {sample_ids}")
    
    # Verify labels are also consistent
    pitt_val_df_sorted = pitt_val_df.sort_values('session_id')
    pitt_best_val_df_sorted = pitt_best_val_df.sort_values('session_id')
    pitt_residual_val_df_sorted = pitt_residual_val_df.sort_values('session_id')
    
    labels_match = (
        list(pitt_val_df_sorted['ad']) == list(pitt_best_val_df_sorted['ad']) and
        list(pitt_best_val_df_sorted['ad']) == list(pitt_residual_val_df_sorted['ad'])
    )
    
    if labels_match:
        print("✓ Labels (AD status) are also consistent across datasets!")
    else:
        print("⚠ WARNING: Labels differ across datasets!")
else:
    print("✗ ERROR: Datasets use DIFFERENT validation sets!")
    print(f"  Pitt only: {pitt_sessions - pitt_best_sessions - pitt_residual_sessions}")
    print(f"  Pitt-Best only: {pitt_best_sessions - pitt_sessions - pitt_residual_sessions}")
    print(f"  Pitt-residual-Best only: {pitt_residual_sessions - pitt_sessions - pitt_best_sessions}")

In [ ]:
# Test on ADReSS dataset
adress_results = test_on_dataset(
    dataset_name="ADReSS",
    model=test_model,
    device=device,
    ssl_model=ssl_model,
    raw_audio_dir="data/raw/ADReSS"
)

In [ ]:
# Test on Lu dataset
lu_results = test_on_dataset(
    dataset_name="Lu",
    model=test_model,
    device=device,
    ssl_model=ssl_model,
    raw_audio_dir="data/raw/Lu"
)

In [ ]:
# Collect all test results
all_test_results = {
    'Pitt': pitt_results,
    'Pitt-Best': pitt_best_results,
    'Pitt-residual-Best': pitt_residual_results,
    'ADReSS': adress_results,
    'Lu': lu_results
}

# Print summary table
print("\n" + "="*80)
print("SUMMARY OF ALL TEST RESULTS")
print("="*80)
print(f"{'Dataset':<20} {'Accuracy':<12} {'F1 Score':<12} {'Control Acc':<12} {'Dementia Acc':<12}")
print("-"*80)

for dataset_name, results in all_test_results.items():
    print(f"{dataset_name:<20} {results['accuracy']*100:>10.2f}%  {results['f1']:>10.4f}  "
          f"{results['control_acc']*100:>10.2f}%  {results['dementia_acc']*100:>10.2f}%")

print("="*80)

In [ ]:
torch.cuda.empty_cache()